In [ ]:
import ee, eemont
import geemap
import geemap.colormaps as geecm

In [ ]:
import pandas as pd

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
save_dir = "to-complete-with-local-dir/"

# 1. Datasets

In [ ]:
fc_lia = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025')

In [ ]:
fc_buffer_only = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_buffer200_only')

In [ ]:
carto_lc = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25")

In [ ]:
carto_lc_no_buffer = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_no_buffer")

In [ ]:
rf_proba = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_rf_only_proba")

In [ ]:
validation_scores = pd.read_csv(os.path.join(save_dir, "rf_validation_scores_g25.csv"), index_col=0)

In [ ]:
scale = 20
crs="EPSG:3035"

# 2. Processing

In [ ]:
rf_proba = rf_proba.updateMask(carto_lc.neq(4)).updateMask(carto_lc.neq(5)).updateMask(carto_lc.neq(6))

In [ ]:
carto_lc = carto_lc.updateMask(carto_lc.neq(4)).updateMask(carto_lc.neq(5)).updateMask(carto_lc.neq(6))

In [ ]:
carto_lc_no_buffer = carto_lc_no_buffer.updateMask(carto_lc_no_buffer.neq(4)).updateMask(carto_lc_no_buffer.neq(5)).updateMask(carto_lc_no_buffer.neq(6))

Get carto with buffer only

In [ ]:
buffer_image = fc_buffer_only.reduceToImage(properties=['FID'], reducer=ee.Reducer.first())

In [ ]:
carto_lc_buffer_only = carto_lc.updateMask(buffer_image.mask()==1).updateMask(carto_lc.neq(4)).updateMask(carto_lc.neq(5)).updateMask(carto_lc.neq(6))

# 3. Uncertainties

## 3.1 Confidence level - predictive set

In [ ]:
q_score = validation_scores.quantile(q=0.05).iloc[0] # 95% confidence level

In [ ]:
predictive_set = rf_proba.where(rf_proba<q_score, 0) 
predictive_set = predictive_set.where(predictive_set.gt(0), 1)

In [ ]:
predictive_set_sum = predictive_set.reduce("sum") 

## 3.2 Map uncertainty

In [ ]:
carto_uncert = predictive_set_sum.updateMask(predictive_set_sum.eq(1)) 

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(carto_uncert, 9)

paramVis_classif = {
  "min": -1,
  "max": 6,
  "palette": [
    '#ffffff',
    '#a5763b',
    'lightgreen',
    'purple',
    'darkgreen',
    'lightblue',
    'blue',
    'yellow']
}

Map.addLayer(carto_lc, paramVis_classif)
Map.addLayer(carto_lc_no_buffer, paramVis_classif)
Map.addLayer(carto_lc_buffer_only, paramVis_classif)
Map.addLayer(carto_uncert, {"min":0, "max":1})
Map.addLayer(carto_uncert_to_save, {"min":0, "max":1})

Map

### Download

Takes about 10 mins

In [ ]:
geemap.ee_export_image_to_asset(carto_uncert.rename(["uncertainty"]),
                                assetId="users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_mask_uncert",
                                scale=scale,
                                crs=crs,
                                maxPixels=1e9,
                                region=fc_lia.geometry().convexHull())

Locallly

In [ ]:
carto_uncert = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_mask_uncert")

In [ ]:
# Change mask to conserve only uncertain pixel with value 0 - all other masked
carto_uncert_to_save = carto_uncert.unmask()
carto_uncert_to_save = carto_uncert_to_save.updateMask(carto_uncert_to_save.eq(0))
carto_uncert_to_save = carto_uncert_to_save.updateMask(carto_lc.neq(4)).updateMask(carto_lc.neq(5)).updateMask(carto_lc.neq(6))

In [ ]:
fc_lia_buffer_deiced = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_with_buffer200_deiced')

In [ ]:
geemap.ee_export_image_to_drive(carto_uncert_to_save,
                                folder="g25_final",
                                fileNamePrefix="carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_mask_uncert",
                                scale=scale,
                                crs=crs,
                                maxPixels=1e10,
                                region=fc_lia_buffer_deiced.geometry())

## 3.2 Uncertainty surfaces

In [ ]:
carto_uncert = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_mask_uncert")

### Functions

In [ ]:
def get_uncert_neg(lc_uncertain, lc_codes=[0,1,2,3], lc_types=["rocks", "grass", "shrubs", "forest"], geom=None, scale=None, crs=None): 
    
    uncert_neg = []
    
    for lc_c, lc_t in zip(lc_codes, lc_types):
        pixelArea = ee.Image.pixelArea()
        uncert_neg.append(pixelArea.updateMask(lc_uncertain.eq(lc_c)).reduceRegion(reducer=ee.Reducer.sum(),
                                                                                    geometry=geom,
                                                                                    scale=scale,
                                                                                    crs=crs,
                                                                                    maxPixels=1e9).rename(["area"], [lc_t]))
    
    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([0,0]), feat) for feat in uncert_neg]) #Fake geometries to be able to export as FeatureCollection

In [ ]:
def get_uncert_pos(predictive_set, carto_lc, lc_codes=[0,1,2,3], lc_types=["rocks", "grass", "shrubs", "forest"], geom=None, scale=None, crs=None):

    uncert_pos = []
    
    for lc_c, lc_t in zip(lc_codes, lc_types):
        pixelArea = ee.Image.pixelArea()
        uncert_pos.append(pixelArea.updateMask(predictive_set.select(lc_t).updateMask(carto_lc.neq(lc_c))
                                               ).reduceRegion(reducer=ee.Reducer.sum(),
                                                               geometry=geom,
                                                               scale=scale,
                                                               crs=crs,
                                                               maxPixels=1e9).rename(["area"], [lc_t]))

    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([0,0]), feat) for feat in uncert_pos]) #Fake geometries to be able to export as FeatureCollection

### 3.2.1 Negative uncert

#### 1. LIA

In [ ]:
lc_uncertain_lia = carto_lc_no_buffer.updateMask(carto_uncert.unmask().neq(1))

In [ ]:
uncert_neg_lia = get_uncert_neg(lc_uncertain_lia, 
                                lc_codes=[0,1,2,3], 
                                lc_types=["rocks", "grass", "shrubs", "forest"], 
                                geom=fc_lia.geometry().convexHull(), 
                                scale=scale, 
                                crs=crs)

In [ ]:
# Takes about 1 min
geemap.ee_export_vector_to_asset(uncert_neg_lia, 
                                 assetId='users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_uncert_neg_lia')

#### 2. Buffer

In [ ]:
lc_uncertain_buffer = carto_lc_buffer_only.updateMask(carto_uncert.unmask().neq(1))

In [ ]:
uncert_neg_buffer = get_uncert_neg(lc_uncertain_buffer, 
                                   lc_codes=[0,1,2,3], 
                                   lc_types=["rocks", "grass", "shrubs", "forest"], 
                                   geom=fc_buffer_only.geometry().convexHull(), 
                                   scale=scale, 
                                   crs=crs)

In [ ]:
# Takes about 1 min
geemap.ee_export_vector_to_asset(uncert_neg_buffer, 
                                 assetId='users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_uncert_neg_buffer')

### 3.2.2 Positive uncert

#### 1. LIA

In [ ]:
uncert_pos_lia = get_uncert_pos(predictive_set,
                                carto_lc_no_buffer,
                                lc_codes=[0,1,2,3], 
                                lc_types=["rocks", "grass", "shrubs", "forest"], 
                                geom=fc_lia.geometry().convexHull(), 
                                scale=scale, 
                                crs=crs)

In [ ]:
geemap.ee_export_vector_to_asset(uncert_pos_lia, 
                                 assetId='users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_uncert_pos_lia')

#### 2. Buffer

In [ ]:
uncert_pos_buffer = get_uncert_pos(predictive_set, 
                                   carto_lc_buffer_only,
                                   lc_codes=[0,1,2,3], 
                                   lc_types=["rocks", "grass", "shrubs", "forest"], 
                                   geom=fc_buffer_only.geometry().convexHull(), 
                                   scale=scale, 
                                   crs=crs)

In [ ]:
geemap.ee_export_vector_to_asset(uncert_pos_buffer, 
                                 assetId='users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/carto_h1b_uncert_pos_buffer')